# 02. Multi-Agent Orchestrations - Microsoft Agent Framework Edition

## 개요
이 노트북에서는 Microsoft Agent Framework를 사용하여 멀티 에이전트 시스템을 구현합니다.

### 학습 목표
- 여러 에이전트를 조율하는 방법
- 에이전트 간 협업 패턴
- 작업 분배 및 결과 통합
- 전문가 에이전트 시스템 구축

## 1. 환경 설정

In [1]:
import os
import asyncio
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import DefaultAzureCredential

# 환경변수 로드
load_dotenv()

deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
credential = DefaultAzureCredential()

# Azure OpenAI Chat Client 생성
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    credential=credential,
    deployment_name=deployment_name,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


## 2. 전문가 에이전트 생성

각기 다른 전문 분야를 가진 에이전트들을 생성합니다.

In [2]:
# 연구 에이전트
researcher = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Research Specialist. Your role is to:
    - Gather and analyze information
    - Identify key facts and data points
    - Provide well-researched insights
    - Cite sources when possible
    """,
    name="Researcher"
)

# 분석 에이전트
analyst = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Data Analyst. Your role is to:
    - Analyze data and patterns
    - Draw meaningful conclusions
    - Identify trends and correlations
    - Provide statistical insights
    """,
    name="Analyst"
)

# 작성 에이전트
writer = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Technical Writer. Your role is to:
    - Create clear, concise documentation
    - Structure information logically
    - Write for the target audience
    - Ensure readability and clarity
    """,
    name="Writer"
)

# 검토 에이전트
reviewer = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Quality Reviewer. Your role is to:
    - Review content for accuracy
    - Check for completeness
    - Identify areas for improvement
    - Provide constructive feedback
    """,
    name="Reviewer"
)

print("✅ 전문가 에이전트 4개 생성 완료")
print("   - Researcher: 정보 수집 및 연구")
print("   - Analyst: 데이터 분석")
print("   - Writer: 문서 작성")
print("   - Reviewer: 품질 검토")

✅ 전문가 에이전트 4개 생성 완료
   - Researcher: 정보 수집 및 연구
   - Analyst: 데이터 분석
   - Writer: 문서 작성
   - Reviewer: 품질 검토


## 3. 순차적 협업 패턴

에이전트들이 순서대로 작업을 수행하는 파이프라인을 구현합니다.

In [3]:
async def sequential_collaboration(topic: str):
    """
    순차적 협업: Researcher → Analyst → Writer → Reviewer
    """
    print(f"\n{'='*60}")
    print(f"주제: {topic}")
    print(f"{'='*60}\n")
    
    # Step 1: 연구 단계
    print("📚 Step 1: Research Phase")
    research_result = await researcher.run(
        f"Research the following topic and provide key information: {topic}"
    )
    print(f"Researcher: {research_result.text[:200]}...\n")
    
    # Step 2: 분석 단계
    print("📊 Step 2: Analysis Phase")
    analysis_result = await analyst.run(
        f"Analyze this research and identify key patterns:\n\n{research_result.text}"
    )
    print(f"Analyst: {analysis_result.text[:200]}...\n")
    
    # Step 3: 작성 단계
    print("✍️ Step 3: Writing Phase")
    writing_result = await writer.run(
        f"""Write a comprehensive report based on this research and analysis:
        
        Research:
        {research_result.text}
        
        Analysis:
        {analysis_result.text}
        """
    )
    print(f"Writer: {writing_result.text[:200]}...\n")
    
    # Step 4: 검토 단계
    print("🔍 Step 4: Review Phase")
    review_result = await reviewer.run(
        f"Review this report and provide feedback:\n\n{writing_result.text}"
    )
    print(f"Reviewer: {review_result.text[:200]}...\n")
    
    print(f"{'='*60}")
    print("✅ 순차적 협업 완료")
    print(f"{'='*60}\n")
    
    return {
        "research": research_result.text,
        "analysis": analysis_result.text,
        "report": writing_result.text,
        "review": review_result.text
    }

# 실행
results = await sequential_collaboration(
    "The impact of AI on the job market in 2025"
)


주제: The impact of AI on the job market in 2025

📚 Step 1: Research Phase
Researcher: Below is a **research-based overview of the impact of AI on the job market in 2025**, focusing on key trends, data points, and sector-specific effects. Sources are cited where possible.

---

## 1. Ov...

📊 Step 2: Analysis Phase
Analyst: Here is a **data-driven analysis of key patterns and trends** emerging from the research, framed from a labor‑market and economic perspective rather than a narrative summary.

---

## 1. Net Job Growt...

✍️ Step 3: Writing Phase
Writer: # The Impact of Artificial Intelligence on the Job Market in 2025  
**Comprehensive Research and Analysis Report**

---

## Executive Overview

By 2025, artificial intelligence (AI) has become a defin...

🔍 Step 4: Review Phase
Reviewer: Here is a structured quality review of the report, focusing on **accuracy, completeness, strengths, gaps, and actionable improvements**.

---

## Overall Assessment

This is a **high-quality, well-st

## 4. 병렬 협업 패턴

여러 에이전트가 동시에 작업을 수행하고 결과를 통합합니다.

In [4]:
async def parallel_collaboration(topic: str):
    """
    병렬 협업: 여러 전문가가 동시에 작업하고 결과를 통합
    """
    print(f"\n{'='*60}")
    print(f"주제: {topic}")
    print(f"{'='*60}\n")
    
    # 각 에이전트에게 다른 관점에서 분석 요청
    tasks = [
        researcher.run(f"Research technical aspects of: {topic}"),
        analyst.run(f"Analyze market trends related to: {topic}"),
        writer.run(f"Describe use cases and applications of: {topic}")
    ]
    
    # 병렬 실행
    print("🔄 병렬로 작업 실행 중...")
    results = await asyncio.gather(*tasks)
    
    # 결과 출력
    print("\n📚 Technical Research:")
    print(f"{results[0].text[:200]}...\n")
    
    print("📊 Market Analysis:")
    print(f"{results[1].text[:200]}...\n")
    
    print("✍️ Use Cases:")
    print(f"{results[2].text[:200]}...\n")
    
    # 통합 리포트 작성
    print("📝 통합 리포트 작성 중...")
    final_report = await writer.run(
        f"""Create a comprehensive report by integrating these perspectives:
        
        Technical Research:
        {results[0].text}
        
        Market Analysis:
        {results[1].text}
        
        Use Cases:
        {results[2].text}
        """
    )
    
    print(f"\n{'='*60}")
    print("✅ 병렬 협업 및 통합 완료")
    print(f"{'='*60}\n")
    
    return final_report.text

# 실행
integrated_report = await parallel_collaboration(
    "Microsoft Agent Framework"
)


주제: Microsoft Agent Framework

🔄 병렬로 작업 실행 중...

📚 Technical Research:
Below is a **technical research overview of Microsoft Agent Framework**, focusing on architecture, components, protocols, and integration patterns. Where possible, I cite **official Microsoft document...

📊 Market Analysis:
Below is a **market trend analysis of the Microsoft Agent Framework**, positioned within the broader **AI agent, enterprise automation, and cloud ecosystem**.

---

## 1. Market Context

The **Microso...

✍️ Use Cases:
Below is a structured overview of the **use cases and applications of the Microsoft Agent Framework**, written for technical and business audiences seeking to understand where and how it can be applie...

📝 통합 리포트 작성 중...

✅ 병렬 협업 및 통합 완료



## 5. 조정자 패턴 (Coordinator Pattern)

중앙 조정자가 다른 에이전트들의 작업을 관리합니다.

In [5]:
# 조정자 에이전트 생성
coordinator = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Project Coordinator. Your role is to:
    - Break down complex tasks into subtasks
    - Assign tasks to appropriate specialists
    - Coordinate work between team members
    - Ensure project completion
    - Provide clear, actionable instructions
    """,
    name="Coordinator"
)

async def coordinated_workflow(project_description: str):
    """
    조정자 패턴: Coordinator가 작업을 분배하고 관리
    """
    print(f"\n{'='*60}")
    print(f"프로젝트: {project_description}")
    print(f"{'='*60}\n")
    
    # Step 1: 조정자가 작업 분석 및 계획
    print("🎯 Step 1: Project Planning")
    plan_result = await coordinator.run(
        f"""Analyze this project and create a task breakdown:
        Project: {project_description}
        
        Provide:
        1. Task breakdown
        2. Assignment to specialists (Researcher, Analyst, Writer)
        3. Execution order
        """
    )
    print(f"Coordinator Plan:\n{plan_result.text}\n")
    
    # Step 2: 연구자에게 작업 할당
    print("📚 Step 2: Research Phase")
    research_task = await coordinator.run(
        f"""Based on the plan, what specific research task should be given to the Researcher?
        Provide only the task description.
        
        Plan: {plan_result.text}
        """
    )
    research_result = await researcher.run(research_task.text)
    print(f"Research Result: {research_result.text[:200]}...\n")
    
    # Step 3: 분석자에게 작업 할당
    print("📊 Step 3: Analysis Phase")
    analysis_task = await coordinator.run(
        f"""Based on the research results, what analysis task should be given to the Analyst?
        
        Research: {research_result.text}
        """
    )
    analysis_result = await analyst.run(analysis_task.text)
    print(f"Analysis Result: {analysis_result.text[:200]}...\n")
    
    # Step 4: 작성자에게 최종 보고서 작성 지시
    print("✍️ Step 4: Report Writing")
    writing_task = await coordinator.run(
        f"""Create final report instructions for the Writer based on all results.
        
        Research: {research_result.text}
        Analysis: {analysis_result.text}
        """
    )
    final_report = await writer.run(writing_task.text)
    print(f"Final Report: {final_report.text[:200]}...\n")
    
    print(f"{'='*60}")
    print("✅ 조정된 워크플로우 완료")
    print(f"{'='*60}\n")
    
    return final_report.text

# 실행
final_output = await coordinated_workflow(
    "Create a comprehensive guide on implementing AI agents in enterprise environments"
)


프로젝트: Create a comprehensive guide on implementing AI agents in enterprise environments

🎯 Step 1: Project Planning
Coordinator Plan:
Below is a structured project breakdown for creating a **comprehensive guide on implementing AI agents in enterprise environments**, aligned with your requested outputs and optimized for coordinated execution.

---

## 1. Task Breakdown

### Phase 1: Project Scoping & Framework Definition
1. Define target audience (e.g., CIOs, IT managers, architects, business leaders)
2. Define scope and depth (technical vs strategic balance)
3. Define key objectives and success criteria for the guide
4. Create a high-level outline and chapter structure

---

### Phase 2: Research & Information Gathering
5. Research enterprise AI agent concepts and definitions  
6. Research common AI agent architectures (e.g., autonomous agents, multi-agent systems, tool-using agents)
7. Research enterprise use cases (IT ops, customer service, RPA, knowledge management, decision suppor

## 6. 반복적 개선 패턴

에이전트들이 피드백을 주고받으며 결과를 개선합니다.

In [ ]:
async def iterative_improvement(topic: str, max_iterations: int = 3):
    """
    반복적 개선: Writer와 Reviewer가 피드백 루프를 형성
    """
    print(f"\n{'='*60}")
    print(f"주제: {topic}")
    print(f"최대 반복: {max_iterations}회")
    print(f"{'='*60}\n")
    
    # 초기 작성
    print("✍️ 초기 문서 작성 중...")
    current_draft = await writer.run(
        f"Write a technical article about: {topic}"
    )
    print(f"Draft 1: {current_draft.text[:150]}...\n")
    
    # 반복적 개선
    for iteration in range(max_iterations):
        print(f"🔄 Iteration {iteration + 1}/{max_iterations}")
        
        # 검토
        review = await reviewer.run(
            f"""Review this article and provide specific improvement suggestions:
            
            {current_draft.text}
            
            Focus on:
            - Clarity
            - Completeness
            - Technical accuracy
            - Structure
            """
        )
        print(f"Review: {review.text[:150]}...")
        
        # 개선
        improved_draft = await writer.run(
            f"""Improve the article based on this feedback:
            
            Original Article:
            {current_draft.text}
            
            Feedback:
            {review.text}
            """
        )
        print(f"Improved: {improved_draft.text[:150]}...\n")
        
        current_draft = improved_draft
    
    print(f"{'='*60}")
    print(f"✅ {max_iterations}회 반복 개선 완료")
    print(f"{'='*60}\n")
    
    return current_draft.text

# 실행
final_article = await iterative_improvement(
    "Best practices for multi-agent system design",
    max_iterations=2
)


주제: Best practices for multi-agent system design
최대 반복: 2회

✍️ 초기 문서 작성 중...
Draft 1: # Best Practices for Multi‑Agent System Design

## Introduction

Multi‑agent systems (MAS) consist of multiple autonomous entities (agents) that perce...

🔄 Iteration 1/2
Review: Below is a structured quality review with **specific, actionable improvement suggestions**, organized around the four requested focus areas.

---

# O...
Improved: Below is a **revised and improved version of the article**, incorporating the feedback while preserving its strengths. The content remains concise but...

🔄 Iteration 2/2
Review: Below is a structured quality review with **specific, actionable improvement suggestions**, organized around **clarity, completeness, technical accura...


## 7. 전문가 합의 패턴

여러 전문가의 의견을 수렴하여 최종 결론을 도출합니다.

In [ ]:
async def expert_consensus(question: str):
    """
    전문가 합의: 여러 전문가의 의견을 종합
    
    주의: 이 함수를 실행하기 전에 Section 2의 에이전트 정의 셀을 먼저 실행해주세요.
    """
    # 에이전트가 정의되지 않았을 경우 재정의
    try:
        # 에이전트 존재 여부 확인
        researcher.name
    except NameError:
        print("⚠️ 에이전트가 정의되지 않았습니다. 재정의 중...\n")
        
        # 에이전트 재정의
        researcher = ChatAgent(
            chat_client=chat_client,
            instructions="""
            You are a Research Specialist. Your role is to:
            - Gather and analyze information
            - Identify key facts and data points
            - Provide well-researched insights
            - Cite sources when possible
            """,
            name="Researcher"
        )
        
        analyst = ChatAgent(
            chat_client=chat_client,
            instructions="""
            You are a Data Analyst. Your role is to:
            - Analyze data and patterns
            - Draw meaningful conclusions
            - Identify trends and correlations
            - Provide statistical insights
            """,
            name="Analyst"
        )
        
        writer = ChatAgent(
            chat_client=chat_client,
            instructions="""
            You are a Technical Writer. Your role is to:
            - Create clear, concise documentation
            - Structure information logically
            - Write for the target audience
            - Ensure readability and clarity
            """,
            name="Writer"
        )
        
        coordinator = ChatAgent(
            chat_client=chat_client,
            instructions="""
            You are a Project Coordinator. Your role is to:
            - Break down complex tasks into subtasks
            - Assign tasks to appropriate specialists
            - Coordinate work between team members
            - Ensure project completion
            - Provide clear, actionable instructions
            """,
            name="Coordinator"
        )
    
    print(f"\n{'='*60}")
    print(f"질문: {question}")
    print(f"{'='*60}\n")
    
    # 각 전문가의 의견 수집
    print("💭 전문가 의견 수집 중...\n")
    
    opinions = []
    agents = [
        (researcher, "Research perspective"),
        (analyst, "Analytical perspective"),
        (writer, "Communication perspective")
    ]
    
    for agent, perspective in agents:
        opinion = await agent.run(
            f"From a {perspective}, answer this question: {question}"
        )
        opinions.append((agent.name, opinion.text))
        print(f"{agent.name}: {opinion.text[:150]}...\n")
    
    # 합의 도출
    print("🤝 합의 도출 중...")
    consensus = await coordinator.run(
        f"""Synthesize these expert opinions into a consensus answer:
        
        Question: {question}
        
        {chr(10).join([f"{name}: {opinion}" for name, opinion in opinions])}
        
        Provide:
        1. Common agreement points
        2. Key insights from each perspective
        3. Balanced final conclusion
        """
    )
    
    print(f"\n{'='*60}")
    print("✅ 전문가 합의 도출 완료")
    print(f"{'='*60}\n")
    print(f"Final Consensus:\n{consensus.text}")
    
    return consensus.text

# 실행
consensus_result = await expert_consensus(
    "What are the key considerations when choosing between single-agent and multi-agent systems?"
)

⚠️ 에이전트가 정의되지 않았습니다. 재정의 중...


질문: What are the key considerations when choosing between single-agent and multi-agent systems?

💭 전문가 의견 수집 중...

Researcher: When deciding between single-agent and multi-agent systems (MAS), several key considerations should be taken into account, each of which influences th...

Analyst: When deciding between single-agent and multi-agent systems, several key considerations should be taken into account from an analytical perspective:

1...

Writer: When choosing between single-agent and multi-agent systems from a communication perspective, several key considerations must be taken into account. He...

🤝 합의 도출 중...

✅ 전문가 합의 도출 완료

Final Consensus:
## 1. Common Agreement Points:

- **Task Complexity**: All perspectives agree that single-agent systems are better suited for simple, well-defined tasks, while multi-agent systems are necessary for complex tasks requiring collaboration and negotiation.
- **Scalability**: There is a consensus that single-agent s

## 마무리

### 학습한 내용
1. ✅ 전문가 에이전트 시스템 구축
2. ✅ 순차적 협업 패턴 (Pipeline)
3. ✅ 병렬 협업 패턴 (Parallel Processing)
4. ✅ 조정자 패턴 (Coordinator)
5. ✅ 반복적 개선 패턴 (Iterative Refinement)
6. ✅ 전문가 합의 패턴 (Expert Consensus)

### 다음 단계
- `03_Workflow.ipynb`: Graph-based 워크플로우
- `04_Research.ipynb`: 웹 검색 통합

### 핵심 개념
- **분업**: 각 에이전트는 특정 전문 분야를 담당
- **협업**: 에이전트들이 정보를 공유하고 함께 작업
- **조율**: 작업 흐름을 관리하는 조정 메커니즘
- **통합**: 여러 관점을 하나의 결과로 통합